<a href="https://colab.research.google.com/github/abxda/aurora-estadistica-descriptiva/blob/main/Practica_1_Estadistica_Descriptiva_Aurora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ☕ Práctica 1 — Estadística Descriptiva con Aurora Coffee

**Estadística Descriptiva · Universidad Aurora · Licenciatura · Sesión 1**

Este cuaderno es el **material completo de la Sesión 1**. Reproduce, paso a paso y con todo
el código resuelto, lo que vemos en clase: cómo pasar de una tabla de datos cruda a un puñado
de números que describen la realidad de una cafetería y permiten **tomar decisiones**.

> Está pensado para que lo entiendas **leyéndolo solo**, aunque hayas faltado a clase. Cada
> sección explica primero la idea y luego la ejecuta en Python. No hay nada que completar:
> solo ejecuta las celdas en orden (▶ o `Shift + Enter`) y lee las explicaciones.

**Mapa de la sesión (lo mismo que las diapositivas):**
1. Población y muestra · Parámetro y estadístico
2. Tipos de variable
3. Tabla de frecuencias
4. Visualizar cualidades (barras y pastel)
5. Tendencia central (media, mediana, moda)
6. Media vs. mediana: el efecto de los valores atípicos
7. Dispersión (rango, varianza, desviación estándar)
8. Posición y forma (cuartiles y boxplot)
9. El retrato completo de una variable


## Paso 0 · Preparar los datos

Antes de nada cargamos las librerías y **creamos la tabla de Aurora Coffee**. Los datos se
generan aquí mismo (no hay que descargar nada). Usamos una *semilla* (`seed`) para que todos
obtengamos exactamente los mismos números y los resultados sean reproducibles.

Cada fila es un **ticket de venta** con seis variables:
`sucursal`, `metodo_pago`, `satisfaccion`, `peso_g` (peso de la bolsa de café),
`cafes_dia` (cafés vendidos ese día) y `propina`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(7)  # reproducibilidad: todos obtienen los mismos datos
n = 200

sucursal     = np.random.choice(["Centro", "Norte", "Sur"], size=n, p=[0.45, 0.35, 0.20])
metodo_pago  = np.random.choice(["Efectivo", "Tarjeta", "App"], size=n, p=[0.30, 0.45, 0.25])
satisfaccion = np.random.choice(["Baja", "Media", "Alta"], size=n, p=[0.15, 0.45, 0.40])
peso_g       = np.round(np.random.normal(499.4, 2.1, size=n), 1)      # CONTINUA: se mide
cafes_dia    = np.random.poisson(95, size=n)                          # DISCRETA: se cuenta
propina      = np.round(np.random.gamma(2.0, 6.0, size=n), 2)         # sesgada a la derecha
propina[[12, 130]] = [220.0, 180.0]   # dos clientes excepcionalmente generosos (atípicos)

cafe = pd.DataFrame({
    "sucursal": sucursal, "metodo_pago": metodo_pago, "satisfaccion": satisfaccion,
    "peso_g": peso_g, "cafes_dia": cafes_dia, "propina": propina,
})
print("Datos listos:", cafe.shape[0], "filas (tickets) y", cafe.shape[1], "columnas (variables)")
cafe.head()

## 1 · Población y muestra · Parámetro y estadístico

Toda la estadística empieza con una distinción que hay que tener clarísima:

- **Población:** el universo completo que queremos estudiar. Para Aurora, *todas* las bolsas
  que producirá este año (cientos de miles). Es real pero inalcanzable: no podemos pesarlas
  todas.
- **Muestra:** un subconjunto que sí observamos. Aquí, nuestras 200 filas.

De estos dos objetos salen dos tipos de números:

- **Parámetro:** describe a la **población**. Es fijo, real y casi siempre **desconocido**.
  Se escribe con letras griegas: **μ** (media) y **σ** (desviación).
- **Estadístico:** el mismo número pero calculado sobre la **muestra**. Lo **conocemos** y
  cambia con cada muestra. Se escribe con letras latinas: **x̄** (media) y **s** (desviación).

**La gran idea del curso:** usamos el estadístico (lo que medimos) para estimar el parámetro
(lo que nos importa pero no vemos). Veámoslo con una simulación: creamos una población enorme
con una media "verdadera" conocida y comprobamos que una muestra pequeña la aproxima.

In [ ]:
# Simulamos una POBLACIÓN enorme de pesos con un parámetro VERDADERO conocido
mu_verdadero = 500.0                      # μ (parámetro real, normalmente NO lo conoceríamos)
poblacion = np.random.normal(mu_verdadero, 2.1, size=100_000)

# Tomamos UNA muestra pequeña (como haría control de calidad)
muestra = np.random.choice(poblacion, size=200)

print(f"Parámetro real de la población   μ = {poblacion.mean():.2f} g  (normalmente oculto)")
print(f"Estadístico de nuestra muestra   x̄ = {muestra.mean():.2f} g  (lo que SÍ medimos)")
print(f"\nLa muestra de 200 datos estimó la media real con un error de "
      f"{abs(poblacion.mean()-muestra.mean()):.2f} g.")

🔍 **Qué observar:** la media de la muestra (x̄) cae muy cerca de la media real (μ) aunque
solo miramos 200 de 100,000 datos. *Esa* es la promesa de la estadística: una buena muestra
representa a la población. En las Sesiones 2 y 3 le pondremos número a ese "muy cerca".

## 2 · Tipos de variable

El **tipo** de cada variable decide qué podemos hacer con ella. Es la pregunta que más errores
evita en la práctica. El árbol es:

- **Cualitativas** (categorías, no se promedian):
  - *Nominales*: sin orden natural → `sucursal`, `metodo_pago`.
  - *Ordinales*: con orden, pero sin distancia medible → `satisfaccion` (Baja < Media < Alta).
- **Cuantitativas** (números con los que sí se opera):
  - *Discretas*: se **cuentan**, valores separados → `cafes_dia`.
  - *Continuas*: se **miden**, cualquier valor de un rango → `peso_g`.

⚠️ **Mito a destruir:** que una variable tenga decimales NO la hace continua. El precio
\$19.99 tiene decimales y es discreto (no existe \$19.995). La prueba real: ¿entre dos
valores cualesquiera siempre cabe otro? Si sí, es continua.

Python *infiere* un tipo de dato (`object`/`str` = texto, `int`/`float` = número), pero
**tú** decides la clasificación estadística.

In [ ]:
# Lo que infirió Python:
cafe.info()

In [ ]:
# La clasificación ESTADÍSTICA correcta (la que importa para elegir herramientas):
clasificacion = {
    "sucursal":     "cualitativa nominal",
    "metodo_pago":  "cualitativa nominal",
    "satisfaccion": "cualitativa ordinal",   # tiene orden: Baja < Media < Alta
    "peso_g":       "cuantitativa continua", # se mide
    "cafes_dia":    "cuantitativa discreta", # se cuenta
    "propina":      "cuantitativa continua", # se mide (dinero, pero la tratamos como continua)
}
for var, tipo in clasificacion.items():
    print(f"  {var:14s} -> {tipo}")

🔍 **Qué observar:** `peso_g` y `cafes_dia` son ambas numéricas, pero `peso_g` se **mide**
(continua) y `cafes_dia` se **cuenta** (discreta). A una cualitativa le haremos tablas y
barras; a una cuantitativa, medias e histogramas. Equivocar el tipo es equivocar el análisis.

## 3 · Del caos al orden: la tabla de frecuencias

Para resumir una variable **cualitativa**, contamos cuántas veces aparece cada categoría.
Hay tres frecuencias que conviene distinguir:

- **Absoluta (fᵢ):** el conteo crudo. ¿Cuántos tickets de cada tipo?
- **Relativa (hᵢ):** la proporción del total (fᵢ ÷ n). Mucho más útil para comunicar: "45%"
  se entiende al instante; "90" obliga a preguntar "¿de cuántos?".
- **Acumulada:** la suma progresiva (útil cuando las categorías tienen orden).

En pandas, `value_counts()` hace el conteo en una línea.

In [ ]:
# Tabla de frecuencias del MÉTODO DE PAGO
abs_ = cafe["metodo_pago"].value_counts()
rel_ = cafe["metodo_pago"].value_counts(normalize=True)

tabla_pago = pd.DataFrame({
    "frec_absoluta": abs_,
    "frec_relativa": rel_.round(3),
    "porcentaje":    (rel_ * 100).round(1),
})
tabla_pago

In [ ]:
# La misma tabla para SUCURSAL
abs_s = cafe["sucursal"].value_counts()
rel_s = cafe["sucursal"].value_counts(normalize=True)
tabla_suc = pd.DataFrame({
    "frec_absoluta": abs_s,
    "frec_relativa": rel_s.round(3),
    "porcentaje":    (rel_s * 100).round(1),
})
tabla_suc

🔍 **Qué observar:** la columna `porcentaje` te dice de un vistazo el peso de cada
categoría sin importar el tamaño de la muestra. La tabla de frecuencias es el **puente
obligado**: casi cualquier gráfico que hagamos después se construye a partir de ella.

## 4 · Visualizar cualidades

El ojo entiende formas más rápido que números. Para variables cualitativas:

- **Gráfico de barras:** la opción segura. El ojo compara longitudes con precisión. Si dudas,
  usa barras.
- **Gráfico de pastel:** muestra *partes de un todo*. Solo si son pocas categorías que suman
  100%.
- **Pictograma:** íconos para divulgación; vistoso pero poco preciso.

Hacemos un gráfico de barras por sucursal y por satisfacción, y un pastel del método de pago.

In [ ]:
conteo = cafe["sucursal"].value_counts()

plt.figure(figsize=(6, 3.5))
plt.bar(conteo.index, conteo.values, color=["#0097A7", "#7B1FA2", "#4285F4"])
plt.title("Ventas por sucursal — Aurora Coffee")
plt.xlabel("Sucursal"); plt.ylabel("Número de tickets")
for i, v in enumerate(conteo.values):
    plt.text(i, v + 0.5, str(v), ha="center", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Satisfacción del cliente (variable ORDINAL: respetamos el orden Baja < Media < Alta)
orden = ["Baja", "Media", "Alta"]
conteo_sat = cafe["satisfaccion"].value_counts().reindex(orden)

plt.figure(figsize=(6, 3.5))
plt.bar(conteo_sat.index, conteo_sat.values, color=["#C62828", "#E67E00", "#2E7D32"])
plt.title("Satisfacción de clientes — Aurora Coffee")
plt.xlabel("Nivel"); plt.ylabel("Clientes")
for i, v in enumerate(conteo_sat.values):
    plt.text(i, v + 0.5, str(v), ha="center", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Pastel del método de pago (pocas categorías que suman 100% -> válido)
conteo_pago = cafe["metodo_pago"].value_counts()

plt.figure(figsize=(4.5, 4.5))
plt.pie(conteo_pago.values, labels=conteo_pago.index, autopct="%1.1f%%",
        colors=["#0097A7", "#7B1FA2", "#4285F4"], startangle=90)
plt.title("Método de pago")
plt.tight_layout(); plt.show()

🔍 **Qué observar:** en la variable ordinal `satisfaccion` respetamos el orden natural
(Baja → Media → Alta) con `reindex`, no el orden por frecuencia. La elección del gráfico es
una decisión de **comunicación**: barras para comparar, pastel para una composición simple.

## 5 · El centro de una variable numérica

Si tuvieras que resumir una variable cuantitativa en **un** número, usarías una **medida de
tendencia central**. Hay tres:

- **Media (x̄):** el promedio aritmético (suma ÷ n). Usa todos los datos, pero los extremos
  la arrastran.
- **Mediana:** el valor de en medio al ordenar. Parte los datos en dos mitades. Es **robusta**
  a los extremos.
- **Moda:** el valor que más se repite. Es la única que también sirve para cualitativas.

Las calculamos para el peso de las bolsas.

In [ ]:
media   = cafe["peso_g"].mean()
mediana = cafe["peso_g"].median()
moda    = cafe["peso_g"].mode()[0]

print(f"Media   (x̄): {media:.2f} g")
print(f"Mediana    : {mediana:.2f} g")
print(f"Moda       : {moda:.2f} g")

🔍 **Qué observar:** las tres rondan los 500 g y la media y la mediana son casi idénticas.
Cuando media ≈ mediana, los datos son aproximadamente **simétricos**. Cuando difieren mucho,
hay **sesgo** o valores extremos. Eso es justo lo siguiente.

## 6 · Media vs. mediana: el efecto de los valores atípicos

Un **valor atípico** (*outlier*) es un dato muchísimo más grande o más chico que el resto.
La media y la mediana se comportan de forma muy distinta ante ellos:

- La **media** se deja **arrastrar** por el outlier (entra en la suma con todo su peso).
- La **mediana** apenas se mueve (solo le importa la posición, no el valor del extremo).

La variable `propina` tiene dos clientes que dejaron 220 y 180 (los pusimos a propósito).
Calculamos las medidas con y sin ellos.

In [ ]:
print("PROPINA — con los 2 valores atípicos (220 y 180):")
print(f"  Media  : {cafe['propina'].mean():.2f}")
print(f"  Mediana: {cafe['propina'].median():.2f}")
print(f"  Máximo : {cafe['propina'].max():.2f}")

In [ ]:
# Quitamos los atípicos (propinas por debajo de 100) y recalculamos
propinas_normales = cafe.loc[cafe["propina"] < 100, "propina"]

print("PROPINA — sin los atípicos:")
print(f"  Media  : {propinas_normales.mean():.2f}")
print(f"  Mediana: {propinas_normales.median():.2f}")

🔍 **Qué observar:** al quitar los atípicos, la **media** cambia bastante mientras la
**mediana** casi no se mueve. Por eso los ingresos, salarios y precios de vivienda se reportan
con la **mediana**: describe a la persona típica y no la distorsionan unos pocos valores
extremos. Regla práctica: datos sesgados o con outliers → comunica con la mediana.

## 7 · La dispersión: el centro no basta

Dos máquinas pueden producir bolsas con la **misma media** de 500 g y ser muy distintas: una
saca bolsas de 499 a 501 (confiable) y otra de 450 a 550 (un desastre). La media no las
distingue; la **dispersión** sí. Tres medidas, de menos a más informativa:

- **Rango:** máximo − mínimo. Simple, pero solo mira los dos extremos.
- **Varianza (s²):** promedio de las distancias al cuadrado respecto a la media. Queda en
  unidades raras (gramos²).
- **Desviación estándar (s):** la raíz de la varianza → vuelve a las unidades originales. Es
  la que más se usa: "las bolsas se desvían, en típico, unos *s* gramos de su media".

In [ ]:
media_peso = cafe["peso_g"].mean()
rango      = cafe["peso_g"].max() - cafe["peso_g"].min()
varianza   = cafe["peso_g"].var(ddof=1)
desv       = cafe["peso_g"].std(ddof=1)

print(f"Media               : {media_peso:.2f} g")
print(f"Rango (máx - mín)   : {rango:.2f} g  (de {cafe['peso_g'].min()} a {cafe['peso_g'].max()})")
print(f"Varianza (s²)       : {varianza:.2f} g²  (unidades poco intuitivas)")
print(f"Desviación estándar : {desv:.2f} g  (interpretable directamente)")

🔍 **Qué observar:** la desviación estándar (~2 g) nos dice que las bolsas se alejan, en
típico, unos 2 g de la media. La varianza es ese mismo número al cuadrado: correcta pero poco
intuitiva. Por eso casi siempre reportamos la **desviación estándar**. Será protagonista
absoluta en las Sesiones 2 y 3.

## 8 · Posición y forma: cuartiles y boxplot

Las medidas de **posición** dicen dónde cae un dato respecto a los demás:

- **Percentil p:** el valor que deja por debajo al p% de los datos. El percentil 90 del peso
  es el valor que supera el 90% de las bolsas.
- **Cuartiles:** los percentiles 25, 50 y 75 (Q1, Q2=mediana, Q3). Parten los datos en cuatro
  bloques iguales. Entre Q1 y Q3 vive la mitad central.

El **diagrama de caja y bigotes** (*boxplot*) dibuja todo eso de un vistazo: la caja va de Q1
a Q3, la línea de dentro es la mediana, los bigotes marcan el rango típico y los puntos
sueltos son los **outliers**, señalados automáticamente.

In [ ]:
# Cuartiles del peso
q1 = cafe["peso_g"].quantile(0.25)
q2 = cafe["peso_g"].quantile(0.50)
q3 = cafe["peso_g"].quantile(0.75)
print(f"Q1 (25%): {q1:.2f} g")
print(f"Q2 (50%, mediana): {q2:.2f} g")
print(f"Q3 (75%): {q3:.2f} g")
print(f"Rango intercuartílico (Q3 - Q1): {q3 - q1:.2f} g  (dispersión robusta a outliers)")

In [ ]:
# Boxplots: peso (simétrico) vs propina (sesgada, con outliers)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))

ax[0].boxplot(cafe["peso_g"], vert=True, patch_artist=True,
              boxprops=dict(facecolor="#E1F4F6"))
ax[0].set_title("Peso de la bolsa (g)"); ax[0].set_ylabel("gramos")

ax[1].boxplot(cafe["propina"], vert=True, patch_artist=True,
              boxprops=dict(facecolor="#F1E7F8"))
ax[1].set_title("Propina por ticket")
plt.tight_layout(); plt.show()

🔍 **Qué observar:** el boxplot del **peso** es casi simétrico y sin puntos fuera de los
bigotes (datos sanos). El de la **propina** muestra los puntos atípicos arriba (los clientes
generosos) y una caja desplazada hacia abajo: señal clara de **sesgo a la derecha**. Un
boxplot te da centro, dispersión, asimetría y outliers en una sola figura.

## 9 · El retrato completo: el resumen de cinco números

Describir bien una variable numérica nunca es un número; son **tres dimensiones**:
**centro** (media y mediana), **dispersión** (desviación y rango) y **forma/posición**
(cuartiles y outliers). A la combinación mínimo–Q1–mediana–Q3–máximo se le llama el
**resumen de cinco números**, y `describe()` lo entrega de un jalón.

In [ ]:
cafe[["peso_g", "cafes_dia", "propina"]].describe().round(2)

🔍 **Cómo leer esta tabla:**
- `mean` = media · `std` = desviación estándar · `min`/`max` = extremos.
- `25%`, `50%`, `75%` = cuartiles (Q1, mediana, Q3).
- Compara `mean` y `50%` (mediana): si se parecen, la variable es simétrica (caso de `peso_g`);
  si la media es mucho mayor, hay sesgo a la derecha (caso de `propina`, por los atípicos).

Con esta tabla y un boxplot tienes la **ficha de identidad** de cualquier variable. El valor
no está en obtener los números, sino en **leerlos**: saber qué pregunta responde cada uno.

---
## 🎒 Y ahora… tu tarea

Acabas de ver el procedimiento completo. Tu **entregable** es un **reporte en PDF**, descrito
en el documento **«Tarea 1 — Describiendo a Aurora Coffee»**.

En la tarea se te pedirá **ejecutar** algunas de estas celdas, **modificar** algún valor para
explorar (por ejemplo, graficar otra variable), **tomar capturas** de los resultados y
**explicar con tus palabras** lo que observas. Todo ya está aquí; la tarea solo te pide
interpretarlo.

> 💡 Guarda este cuaderno (Archivo → Guardar una copia en Drive) para tenerlo a mano. Es tu
> material de estudio para el examen final.
